# 2.5 Neural Network Fitting Benchmarks

This notebook runs the full benchmark sweep of neural-network SHO fitting:
optimizers (Adam, Trust-Region CG) x noise levels (0-8) x batch sizes x seeds.

Notes:
- The full grid (360 training runs) is designed for a GPU (e.g., Google Colab).
- Noise levels 1-8 require the noisy datasets *and their LSQF SHO fits* generated by
  running notebook `0_5_Noisy_Data_and_Fitting.ipynb` at full scale (`QUICK_RUN = False`).
- With `QUICK_RUN = True` a single small training run on the raw data (noise 0) is executed
  to verify the pipeline end-to-end on a CPU.

In [ ]:
# --- Environment setup (works on Colab and locally) ---
REPO_URL = "https://github.com/m3-learning/m3_learning.git"
BRANCH = "shofit"
QUICK_RUN = False  # True = small subset / few epochs, for fast verification

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import importlib.util, subprocess
    if importlib.util.find_spec("m3_learning") is None:
        subprocess.run(["git", "clone", "-b", BRANCH, "--depth", "1",
                        REPO_URL, "/content/m3_learning_repo"], check=True)
        subprocess.run(["pip", "install", "-q",
                        "/content/m3_learning_repo/m3_learning"], check=True)
        subprocess.run(["pip", "install", "-q", "--upgrade", "numpy_groupies"], check=True)

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Colab: {IN_COLAB}, device: {device}")

In [ ]:
%load_ext autoreload
%autoreload 2

import itertools
import os
from datetime import datetime

import numpy as np

from m3_learning.be.nn import SHO_fit_func_nn, SHO_Model, batch_training
from m3_learning.util.system_info import SystemInfo
from m3_learning.util.file_IO import download_and_unzip
from m3_learning.optimizers.TrustRegion import TRCG
from m3_learning.nn.random import random_seed
from m3_learning.viz.style import set_style
from m3_learning.viz.printing import printer
from m3_learning.be.viz import Viz
from m3_learning.be.dataset import BE_Dataset

# Note: on multi-GPU machines you can pin a specific GPU, e.g.:
# os.environ['CUDA_VISIBLE_DEVICES'] = '1'

print(f'using torch version {torch.__version__}')

printing = printer(basepath='./Figures/')

set_style("printing")
random_seed(seed=42)

In [ ]:
# Download the data file from Zenodo (skipped if the file already exists,
# e.g. if you already ran notebooks 0_5, 1 or 2)
url = 'https://zenodo.org/record/7774788/files/PZT_2080_raw_data.h5?download=1'

# Specify the filename and the path to save the file
filename = 'data_raw.h5'
save_path = './Data'

# download the file
download_and_unzip(filename, url, save_path)

data_path = save_path + "/" + filename

In [ ]:
# Trust-Region CG optimizer configuration (device-agnostic)
optimizer_TR = {"name": "TRCG", "optimizer": TRCG, "radius": 5,
                "device": device, "ADAM_epochs": 2}

if QUICK_RUN:
    # Minimal grid for fast CPU verification: a single Adam run on the raw data.
    # Noise levels 1-8 are excluded because they need the noisy-data LSQF SHO fits
    # produced by running notebook 0_5 at full scale.
    optimizers = ['Adam']
    noise_list = [0]
    batch_size = [1000]
    epochs = [1]
    seed = [41]
    early_stopping_time = 60
    max_train_size = 10000  # subsample the training data
else:
    # Full benchmark grid exactly as run in the paper (360 training runs)
    optimizers = ['Adam', optimizer_TR]
    noise_list = [0, 1, 2, 3, 4, 5, 6, 7, 8]
    batch_size = [500, 1000, 5000, 10000]
    epochs = [5]
    seed = [41, 43, 44, 45, 46]
    early_stopping_time = 60 * 3
    max_train_size = None

basepath_postfix = 'nn_benchmarks_noise'

In [ ]:
# instantiate the dataset object
dataset = BE_Dataset(data_path, SHO_fit_func_LSQF=SHO_fit_func_nn)

# computes the SHO LSQF fit if it has not been performed previously.
# This is cached/idempotent: it returns instantly if you already ran notebook 1 or 2.
dataset.SHO_Fitter(force=False, h5_sho_targ_grp="Raw_Data_SHO_Fit")

# reinstantiate the dataset object after fitting
dataset = BE_Dataset(data_path, SHO_fit_func_LSQF=SHO_fit_func_nn)

# print the contents of the file
dataset.print_be_tree()

# Get the current date and time used to name the benchmark output folder
# Format the date and time in a 'pretty' format (e.g., YYYY-MM-DD_HH-MM-SS)
formatted_datetime = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')

In [ ]:
basepath = f'{formatted_datetime}_{basepath_postfix}'

# records the system information (CPU/GPU) used for the benchmark
system_info = SystemInfo()
cpu_info, gpu_info = system_info.get_system_info()
system_info.save_to_file("Trained Models/" + basepath,
                         "system_info.txt", cpu_info, gpu_info)

In [ ]:
# Generate all combinations
combinations = list(itertools.product(
    optimizers, noise_list, batch_size, epochs, seed))

for i, training in enumerate(combinations):

    optimizer_ = training[0]
    noise_ = training[1]
    seed_ = training[4]

    print(i, optimizer_, noise_, seed_)

In [ ]:
batch_training(dataset, optimizers, noise_list, batch_size, epochs,
               seed,
               write_CSV="Batch_Trainging_SpeedTest.csv",
               basepath=basepath,
               early_stopping_time=early_stopping_time,
               max_train_size=max_train_size)